# BLOQUE 01 – PERFIL ECONOMICO
## Objetivo
Construir capa consolidada ECONOMICA normalizada por unidad espacial.
## Inputs
- denue.shp
- manzana.shp

## Procesos
- Limpieza
- Homologación de CRS
- Join espacial
- Normalización

## Output
1_D3C1_output_1_Económico.gpkg
- 1_D3C1V1_Unidades económicas.shp
- 1_D3C1V1_Mzn_Unidades_Economicas.shp

In [2]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Polygon
from dbfread import DBF
import jenkspy
import json
from jenkspy import jenks_breaks
import time

In [18]:
# Rutas
DENUE_ruta = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\DENUE\denue_inegi_14_.shp"
manzanas_ruta = r"C:\Users\Alejandro\Desktop\DAL_LAB\data_raw\Manzanas\14m.shp"

ruta_salida = r"C:\Users\Alejandro\Desktop\DAL_LAB\outputs\etapa_1\1_D3C1_output_1_Económico.gpkg"

In [3]:
# Cargar los GeoDataFrames
gdf_denue = gpd.read_file(DENUE_ruta)
gdf_manzanas = gpd.read_file(manzanas_ruta)

In [24]:
print(gdf_denue.crs)
print(gdf_manzanas.crs)

GEOGCS["WGS84(DD)",DATUM["WGS84",SPHEROID["WGS84",6378137,298.257223563]],PRIMEM["Greenwich",0],UNIT["degree",0.0174532925199433],AXIS["Longitude",EAST],AXIS["Latitude",NORTH]]
PROJCS["MEXICO_ITRF_2008_LCC",GEOGCS["ITRF2008",DATUM["International_Terrestrial_Reference_Frame_2008",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","1061"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",12],PARAMETER["central_meridian",-102],PARAMETER["standard_parallel_1",17.5],PARAMETER["standard_parallel_2",29.5],PARAMETER["false_easting",2500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]


In [25]:
gdf_denue = gdf_denue.to_crs(gdf_manzanas.crs)

## Tratado Denue

In [13]:
gdf_denue.columns

Index(['id', 'nom_estab', 'codigo_act', 'nombre_act', 'per_ocu', 'cve_ent',
       'cve_mun', 'cve_loc', 'ageb', 'manzana', 'fecha_alta', 'geometry',
       'Num_Emp'],
      dtype='object')

In [12]:
#Borrar columnas DENUE
gdf_denue = gdf_denue.drop(columns=['clee','raz_social', 
        'tipo_vial', 'nom_vial', 'tipo_v_e_1', 'nom_v_e_1',
       'tipo_v_e_2', 'nom_v_e_2', 'tipo_v_e_3', 'nom_v_e_3', 'numero_ext',
       'letra_ext', 'edificio', 'edificio_e', 'numero_int', 'letra_int',
       'tipo_asent', 'nomb_asent', 'tipoCenCom', 'nom_CenCom', 'num_local',
       'cod_postal','entidad',  'municipio',
       'localidad',  'telefono', 'correoelec', 'www',
       'tipoUniEco', 'latitud', 'longitud'])

In [4]:
#Campo empleados

# Definir la función de conversión
def convert_per_ocu_to_value(per_ocu):
    mapping = {
        '0 a 5 personas': 3,
        '6 a 10 personas': 8,
        '11 a 30 personas': 20,
        '31 a 50 personas': 40,
        '51 a 100 personas': 75,
        '101 a 250 personas': 175,
        '251 y más personas': 275
    }
    return mapping.get(per_ocu, None)

# Aplicar la función para crear la nueva columna
gdf_denue['Num_Emp'] = gdf_denue['per_ocu'].apply(convert_per_ocu_to_value)

In [29]:
# Guardar el DENUE con los nuevos campos
gdf_denue.to_file(ruta_salida, layer='1_D3C1V1_Unidades_económicas', driver='GPKG')

## Tratado Manzanas

In [27]:
# Crear un buffer de 10 metros alrededor de las manzanas
gdf_manzanas_buffer = gdf_manzanas.copy()
gdf_manzanas_buffer['geometry'] = gdf_manzanas_buffer.geometry.buffer(10)

# Realizar la unión espacial entre las manzanas y las unidades económicas
gdf_union = gpd.sjoin(gdf_manzanas_buffer, gdf_denue, how="left", predicate="contains")

# Contar el número de unidades económicas dentro de cada manzana
numero_unidades_por_manzana = gdf_union.groupby('CVEGEO').size().reset_index(name='UE')

# Unir los resultados al GeoDataFrame de las manzanas
gdf_manzanas_final = gdf_manzanas.merge(numero_unidades_por_manzana, on='CVEGEO', how='left')

# Mostrar el resultado
gdf_manzanas_final

,CVEGEO,CVE_ENT,CVE_MUN,CVE_LOC,CVE_AGEB,CVE_MZA,AMBITO,TIPOMZA,geometry,UE
0,1400100010226003,14,001,0001,0226,003,Urbana,Típica,"POLYGON ((2406377.75 974961.594, 2406376.127 9...",4
1,1400100010211015,14,001,0001,0211,015,Urbana,Típica,"POLYGON ((2406681.686 975531.995, 2406674.665 ...",1
2,1400100010211011,14,001,0001,0211,011,Urbana,Típica,"POLYGON ((2406061.676 975289.259, 2406069.062 ...",13
3,1400100010207020,14,001,0001,0207,020,Urbana,Típica,"POLYGON ((2405836.806 974518.055, 2405844.876 ...",8
4,1400100010194015,14,001,0001,0194,015,Urbana,Típica,"POLYGON ((2405774.76 975251.892, 2405774.328 9...",5
...,...,...,...,...,...,...,...,...,...,...
140999,141250052012A040,14,125,0052,012A,040,Rural,Típica,"POLYGON ((2450269.737 963843.34, 2450268.438 9...",1
141000,141250052012A041,14,125,0052,012A,041,Rural,Típica,"POLYGON ((2450228.916 963905.119, 2450273.154 ...",1
141001,141250052012A042,14,125,0052,012A,042,Rural,Típica,"POLYGON ((2450273.461 963909.843, 2450228.419 ...",1
141002,1412500640115025,14,125,0064,0115,025,Rural,Típica,"POLYGON ((2445003.343 977547.067, 2444993.892 ...",1


In [28]:
# Guardar el GeoPackage con los nuevos campos
gdf_manzanas_final.to_file(ruta_salida, layer='1_D3C1V1_Mzn_Unidades_Economicas', driver='GPKG')